In [4]:
import time

notebook_start_time = time.time()
print("[Notebook timer started]")

[Notebook timer started]
[Cell executed in 0.002 seconds]


In [5]:



from IPython import get_ipython

_cell_start_time = None


def _pre_run_cell(info):
    global _cell_start_time
    _cell_start_time = time.time()


def _post_run_cell(result):
    global _cell_start_time
    if _cell_start_time is None:
        return
    elapsed = time.time() - _cell_start_time
    print(f"[Cell executed in {elapsed:.3f} seconds]")


_ip = get_ipython()
if _ip is not None:

    _ip.events.register("pre_run_cell", _pre_run_cell)
    _ip.events.register("post_run_cell", _post_run_cell)
else:
    print("[Timing hooks not installed: no active IPython shell detected]")


# Redoing some analysis to align with SQL analysis

## Data Import 
Importing small sample of data to use as a prototype. Here, we will be using python with the pandas library to attain a performance baseline.

In [6]:
import pandas as pd

df = pd.read_csv("../data/processed/trips_clean.csv")
df.head()

C:\Users\danny\AppData\Local\Temp\ipykernel_32736\1924554489.py:3: DtypeWarning: Columns (0: start_station_id, 1: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/trips_clean.csv")


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,A847FADBBC638E45,docked_bike,2020-04-26 17:45:14,2020-04-26 18:12:03,Eckhart Park,86,Lincoln Ave & Diversey Pkwy,152.0,41.8964,-87.6610,41.9322,-87.6586,member
1,5405B80E996FF60D,docked_bike,2020-04-17 17:08:54,2020-04-17 17:17:03,Drake Ave & Fullerton Ave,503,Kosciuszko Park,499.0,41.9244,-87.7154,41.9306,-87.7238,member
2,5DD24A79A4E006F4,docked_bike,2020-04-01 17:54:13,2020-04-01 18:08:36,McClurg Ct & Erie St,142,Indiana Ave & Roosevelt Rd,255.0,41.8945,-87.6179,41.8679,-87.6230,member
3,2A59BBDF5CDBA725,docked_bike,2020-04-07 12:50:19,2020-04-07 13:02:31,California Ave & Division St,216,Wood St & Augusta Blvd,657.0,41.9030,-87.6975,41.8992,-87.6722,member
4,27AD306C119C6158,docked_bike,2020-04-18 10:22:59,2020-04-18 11:15:54,Rush St & Hubbard St,125,Sheridan Rd & Lawrence Ave,323.0,41.8902,-87.6262,41.9695,-87.6547,casual


[Cell executed in 91.607 seconds]
[Cell executed in 91.607 seconds]


In [7]:
df["started_at"] = pd.to_datetime(df["started_at"], utc=True)
df["ended_at"] = pd.to_datetime(df["ended_at"], utc=True)
df["ride_length"] = df["ended_at"] - df["started_at"]
df["ride_length_minutes"] = df["ride_length"].dt.total_seconds() / 60
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,ride_length_minutes
0,A847FADBBC638E45,docked_bike,2020-04-26 17:45:14+00:00,2020-04-26 18:12:03+00:00,Eckhart Park,86,Lincoln Ave & Diversey Pkwy,152.0,41.8964,-87.6610,41.9322,-87.6586,member,0 days 00:26:49,26.816667
1,5405B80E996FF60D,docked_bike,2020-04-17 17:08:54+00:00,2020-04-17 17:17:03+00:00,Drake Ave & Fullerton Ave,503,Kosciuszko Park,499.0,41.9244,-87.7154,41.9306,-87.7238,member,0 days 00:08:09,8.150000
2,5DD24A79A4E006F4,docked_bike,2020-04-01 17:54:13+00:00,2020-04-01 18:08:36+00:00,McClurg Ct & Erie St,142,Indiana Ave & Roosevelt Rd,255.0,41.8945,-87.6179,41.8679,-87.6230,member,0 days 00:14:23,14.383333
3,2A59BBDF5CDBA725,docked_bike,2020-04-07 12:50:19+00:00,2020-04-07 13:02:31+00:00,California Ave & Division St,216,Wood St & Augusta Blvd,657.0,41.9030,-87.6975,41.8992,-87.6722,member,0 days 00:12:12,12.200000
4,27AD306C119C6158,docked_bike,2020-04-18 10:22:59+00:00,2020-04-18 11:15:54+00:00,Rush St & Hubbard St,125,Sheridan Rd & Lawrence Ave,323.0,41.8902,-87.6262,41.9695,-87.6547,casual,0 days 00:52:55,52.916667


[Cell executed in 16.868 seconds]
[Cell executed in 16.868 seconds]


## Making Time and Day cols
Allows for us to perform day and time analysis

In [8]:
df["hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.day_name()

df[["started_at", "hour", "day_of_week"]].head()

,started_at,hour,day_of_week
0,2020-04-26 17:45:14+00:00,17,Sunday
1,2020-04-17 17:08:54+00:00,17,Friday
2,2020-04-01 17:54:13+00:00,17,Wednesday
3,2020-04-07 12:50:19+00:00,12,Tuesday
4,2020-04-18 10:22:59+00:00,10,Saturday


[Cell executed in 2.922 seconds]
[Cell executed in 2.922 seconds]


## Bike type usage
Here, we wanted to find the relationship between bike type and user type. We found the average ride duration for each bike and user combination. This analysis helps identify user preferences for specific bike types and highlights the differences between members and casual riders that we have previously theorised about. 
The shear number of member usage is higher, but when we look at the percentage usage, the figures start to make sense. Members seem to prefer the "classic bike" over the electric more so than the casual users do, this could be down to members being into fitness or simply the bike availability in their residential areas.
The only question we are left with right now is why do only casual users use "docked bikes"?

In [9]:
bike_user_ct = pd.crosstab(df["rideable_type"], df["member_casual"])
print("Bike type x user type (counts):")
print(bike_user_ct)


print("\nBike type x user type (row %):")
bike_pct = bike_user_ct.div(bike_user_ct.sum(axis=1), axis=0).round(3)
print(bike_pct)

print("\nAverage ride length (minutes) by bike type and user type:")

avg_duration_bike_user = (
    df.groupby(["rideable_type", "member_casual"])["ride_length_minutes"]
    .mean()
    .round(1)
    .unstack()
)
print(avg_duration_bike_user)

print("\nInterpretation by bike type:")
for bike in bike_user_ct.index:
    casual_share = bike_pct.loc[bike, "casual"] * 100
    member_share = bike_pct.loc[bike, "member"] * 100
    print(f"- {bike}: {casual_share:.1f}% casual, {member_share:.1f}% member")

Bike type x user type (counts):
member_casual   casual   member
rideable_type                  
classic_bike   2169435  3753423
docked_bike    1635846  1820293
electric_bike  2412331  3013135

Bike type x user type (row %):
member_casual  casual  member
rideable_type                
classic_bike    0.366   0.634
docked_bike     0.473   0.527
electric_bike   0.445   0.555

Average ride length (minutes) by bike type and user type:
member_casual  casual  member
rideable_type                
classic_bike     28.8    14.0
docked_bike      65.0    13.2
electric_bike    17.4    11.0

Interpretation by bike type:
- classic_bike: 36.6% casual, 63.4% member
- docked_bike: 47.3% casual, 52.7% member
- electric_bike: 44.5% casual, 55.5% member
[Cell executed in 14.650 seconds]
[Cell executed in 14.650 seconds]


## Station-based insights
We wanted to find out which stations had the highest numbers of users arriving and departing from them, we can see that "Dearborn St and Erie St" appears one of the focal points, seeing as it is the most popular destination and second most popular starting point. Similar observations can be made for each other station.
We also computed the average ride duration by start station an user type. We filtered this to include stations with 30+ trips to ensure reliability.
Using this information, we can detarmine what areas are residential and which are tourist hotspots among many other things.

In [10]:
for user_type in ["member", "casual"]:
    print(f"\nTop 10 start stations for {user_type}s:")
    top_start = (
        df[df["member_casual"] == user_type]["start_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_start)

    print(f"\nTop 10 end stations for {user_type}s:")
    top_end = (
        df[df["member_casual"] == user_type]["end_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_end)

min_trips_per_station = 30

avg_duration_station = (
    df.groupby(["start_station_name", "member_casual"])["ride_length_minutes"]
    .agg(["count", "mean"])
    .reset_index()
)



avg_duration_station = avg_duration_station[avg_duration_station["count"] >= min_trips_per_station]

print("\nAverage ride length (minutes) by start station and user type (>= 30 trips):")
print(avg_duration_station.sort_values("mean", ascending=False).head(20))


Top 10 start stations for members:
start_station_name
Clark St & Elm St               67300
Kingsbury St & Kinzie St        65141
Wells St & Concord Ln           60565
Wells St & Elm St               53914
Dearborn St & Erie St           52088
St. Clair St & Erie St          51381
Broadway & Barry Ave            51369
Wells St & Huron St             50383
Clinton St & Madison St         50233
Clinton St & Washington Blvd    46662
Name: count, dtype: int64

Top 10 end stations for members:
end_station_name
Clark St & Elm St               68447
Kingsbury St & Kinzie St        65054
Wells St & Concord Ln           62086
Dearborn St & Erie St           53468
Wells St & Elm St               53432
St. Clair St & Erie St          53212
Broadway & Barry Ave            52843
Clinton St & Madison St         51828
Clinton St & Washington Blvd    48774
Wells St & Huron St             48561
Name: count, dtype: int64

Top 10 start stations for casuals:
start_station_name
Streeter Dr & Grand Ave    

## Time of day x day of week bike usage
We could use seaborn to display this information on a heatmap as it is very suited for that, but seeing as we are attempting to set a base line for time, we thought it best to stick with text outputs in case graphics take a significantly long amount of time to display.
Now bike usage by day of week and time is legible. The huge numbers of users visible during the work rush hours on weekdays confirms our earlier hypotheses. The low numbers of rides over the weekend do so as well, likely being mostly tourist use.

In [11]:
#this is basically a text heat map, I wanted to make something easily duplicated in other languages without importing libraries
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cat_type = pd.CategoricalDtype(categories=day_order, ordered=True)
df["day_of_week"] = df["day_of_week"].astype(cat_type)

for user_type in ["member", "casual"]:
    subset = df[df["member_casual"] == user_type]
    pivot = subset.pivot_table(
        index="day_of_week",
        columns="hour",
        values="ride_id",
        aggfunc="count",
        fill_value=0,
    )

    print(f"\nTrips table (day x hour) for {user_type}:")
    print(pivot)

    print(f"\nTop 5 day-hour cells for {user_type}:")
    top_cells = (
        pivot.stack()
        .sort_values(ascending=False)
        .head(5)
        .reset_index(name="trips")
    )
    for _, row in top_cells.iterrows():
        print(f"- {row['day_of_week']} @ {row['hour']}: {row['trips']} trips")


Trips table (day x hour) for member:
hour            0      1     2     3     4      5      6      7      8   \
day_of_week                                                               
Monday        6793   3805  2196  1628  2913  13485  38771  70888  80007   
Tuesday       5673   2836  1576  1234  3046  16106  47408  88419  96855   
Wednesday     6734   3222  1728  1211  2826  15550  46128  86277  96184   
Thursday      7825   3775  1977  1478  2757  14278  42996  81699  92730   
Friday       10645   5889  3229  1985  2891  12899  37585  65497  72981   
Saturday     20892  15360  8778  4649  3077   5139  13146  24632  40817   
Sunday       22566  15912  9567  5383  3613   4527  10488  17809  28557   

hour            9   ...     14     15      16      17      18     19     20  \
day_of_week         ...                                                       
Monday       48740  ...  60255  74115  106719  141572  113821  78628  50772   
Tuesday      54767  ...  58816  77766  118197  15

## Analysing Daily Trends
we can see saturday is extra busy for casual users which supports the theory they are tourists trying to view the city

In [12]:
weekday_counts = (
    df.groupby(["day_of_week", "member_casual"])
    .size()
    .unstack(fill_value=0)
    .reindex(day_order)
)

member_counts = weekday_counts["member"]
casual_counts = weekday_counts["casual"]


print("Trips by day of week and user type (counts):")
print(weekday_counts)

print("\nDay-of-week share of trips by user type (row %):")
weekday_pct = weekday_counts.div(weekday_counts.sum(axis=1), axis=0).round(3)
print(weekday_pct)

print("\nKey weekday patterns:")

for day in weekday_counts.index:
    casual = weekday_counts.loc[day, "casual"]
    member = weekday_counts.loc[day, "member"]
    print(f"- {day}: {casual} casual vs {member} member trips")

Trips by day of week and user type (counts):
member_casual   casual   member
day_of_week                    
Monday          705703  1181595
Tuesday         675710  1299634
Wednesday       705828  1330545
Thursday        758083  1312374
Friday          900968  1239562
Saturday       1344993  1188704
Sunday         1126327  1034437

Day-of-week share of trips by user type (row %):
member_casual  casual  member
day_of_week                  
Monday          0.374   0.626
Tuesday         0.342   0.658
Wednesday       0.347   0.653
Thursday        0.366   0.634
Friday          0.421   0.579
Saturday        0.531   0.469
Sunday          0.521   0.479

Key weekday patterns:
- Monday: 705703 casual vs 1181595 member trips
- Tuesday: 675710 casual vs 1299634 member trips
- Wednesday: 705828 casual vs 1330545 member trips
- Thursday: 758083 casual vs 1312374 member trips
- Friday: 900968 casual vs 1239562 member trips
- Saturday: 1344993 casual vs 1188704 member trips
- Sunday: 1126327 casual vs

## Busy Stations
We can see the busiest stations, ranked on the number of arrivals + departures

In [13]:
departures = df.groupby("start_station_name").size().rename("departures")
arrivals = df.groupby("end_station_name").size().rename("arrivals")


station_activity = (
    departures.to_frame()
    .join(arrivals.to_frame(), how="outer")
    .fillna(0)
)

station_activity["total_activity"] = (
    station_activity["departures"] + station_activity["arrivals"]
)



top_stations = station_activity.sort_values("total_activity", ascending=False).head(10)

print("Top stations by total activity (departures + arrivals):")
print(top_stations)

Top stations by total activity (departures + arrivals):
                          departures  arrivals  total_activity
start_station_name                                            
Streeter Dr & Grand Ave     193318.0  196538.0        389856.0
Clark St & Elm St           108225.0  107045.0        215270.0
Michigan Ave & Oak St       105784.0  107215.0        212999.0
Wells St & Concord Ln       106241.0  106696.0        212937.0
Millennium Park             101526.0  103847.0        205373.0
Theater on the Lake          99425.0  101092.0        200517.0
Wells St & Elm St            91697.0   88811.0        180508.0
Kingsbury St & Kinzie St     89744.0   87120.0        176864.0
Broadway & Barry Ave         84423.0   85985.0        170408.0
Clark St & Armitage Ave      85698.0   83643.0        169341.0
[Cell executed in 8.827 seconds]
[Cell executed in 8.827 seconds]


## Ride lengths by bike type
We can see what the average ride length is for both casuals and members for each type of bike.

In [14]:

valid = df[df["ride_length_minutes"].notna()].copy()

avg_duration_bike_user = (valid.groupby(["rideable_type", "member_casual"])["ride_length_minutes"].mean().round(1).unstack())

print("Average ride length (minutes) by bike type and user type:")
print(avg_duration_bike_user)


Average ride length (minutes) by bike type and user type:
member_casual  casual  member
rideable_type                
classic_bike     28.8    14.0
docked_bike      65.0    13.2
electric_bike    17.4    11.0
[Cell executed in 69.425 seconds]
[Cell executed in 69.425 seconds]


In [15]:


elapsed = time.time() - notebook_start_time
print(f"[Whole notebook executed in {elapsed:.3f} seconds]")

[Whole notebook executed in 509.481 seconds]
[Cell executed in 0.066 seconds]
[Cell executed in 0.066 seconds]
